<a href="https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb
import os

# Retrieve the token safely from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set environment variable or pass it to duckdb/requests if needed
os.environ['HF_TOKEN'] = hf_token

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **One row =** One unique combination of a client (`client_hash_id`) and a content item performance record for a specific calendar month.
* **Time Window:** Iteration and feature verification will run on the mid-panel test month **`2026-03`** (March 2026), keeping the final month (`2026-06`) sealed as the natural outcome window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features:**
  1. `historical_clicks`: Total clicks recorded in the prior observation window.
  2. `avg_impressions_7d`: Rolling average of daily impressions.
  3. `content_type`: Categorical encoding of the content format.
  4. `age_in_days`: Number of days since the content was first published.
  5. `query_diversity_score`: Ratio of unique search queries leading to the content.
* **Label:** `target_traffic_growth_flag` (Binary indicator of whether traffic increases in the subsequent period).
* **Context:** `client_hash_id`, `date`, `month` (identifiers used for grouping and joining, not predictive features).
* **Excluded:** `future_conversion_rate` (Excluded to strictly prevent target data leakage from the outcome window).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import duckdb
import os
from google.colab import userdata

# 1. Retrieve Hugging Face token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Configure/Overwrite DuckDB Secrets Manager for Hugging Face safely
duckdb.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 3. Run verification queries using the correct column name (report_date)
query = """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS active_surviving_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
"""

df_result = duckdb.sql(query).df()
display(df_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_clients,min_date,max_date,active_surviving_rows
0,11694072,65,2026-06-01,2026-06-30,3878937.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Data Limits & Blind Spots:**
  1. **Unbalanced Historical Tracking:** Early client cohorts only contain Search Console (GSC) metrics, while deep GA4 engagement analytics are missing or populated as null.
  2. **Window Overlaps:** Aggregation windows may overlap during mid-month client onboarding syncs, risking temporary data duplication if not filtered by active status flags.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.